# 7. Siamese network: learn whether two images are similar
We'll train a small shared-weight CNN on MNIST digits. A contrastive loss encourages embeddings of same-class digits to be close and different-class digits to be farther apart.

In [ ]:
!pip -q install torch torchvision matplotlib

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
import random, numpy as np, matplotlib.pyplot as plt
torch.manual_seed(7); random.seed(7)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
tfm=transforms.ToTensor()
train_ds=datasets.MNIST("/content/data",train=True,download=True,transform=tfm)
class PairDataset(Dataset):
    def __init__(self, base):
        self.base=base
        self.by_label={i:[] for i in range(10)}
        # Build label-to-index lookup once, rather than scanning the dataset for every class.
        for i, label in enumerate(base.targets.tolist()):
            self.by_label[int(label)].append(i)
    def __len__(self): return len(self.base)
    def __getitem__(self,idx):
        x1,y1=self.base[idx]
        y1=int(y1)
        same=random.random()<.5
        if same:
            y2=y1
        else:
            y2=random.choice([y for y in range(10) if y!=y1])
        j=random.choice(self.by_label[y2])
        x2,_=self.base[j]
        return x1,x2,torch.tensor(0.0 if same else 1.0)
pairs=PairDataset(train_ds)
loader=DataLoader(pairs,batch_size=128,shuffle=True,num_workers=2)

In [ ]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Conv2d(1,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
          nn.Conv2d(16,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.Flatten(),
          nn.Linear(32*7*7,64))
    def forward(self,x): return F.normalize(self.net(x),p=2,dim=1)
class Siamese(nn.Module):
    def __init__(self): super().__init__(); self.encoder=Encoder()
    def forward(self,a,b): return self.encoder(a),self.encoder(b)
def contrastive_loss(a,b,label,margin=1.0):
    d=F.pairwise_distance(a,b)
    return ((1-label)*d.pow(2)+label*F.relu(margin-d).pow(2)).mean()
model=Siamese().to(device)
opt=torch.optim.Adam(model.parameters(),lr=1e-3)
epochs=2  # Increase to 5-10 for better embeddings
for epoch in range(epochs):
    model.train(); total=0
    for a,b,label in loader:
        a,b,label=a.to(device),b.to(device),label.to(device)
        z1,z2=model(a,b); loss=contrastive_loss(z1,z2,label)
        opt.zero_grad(); loss.backward(); opt.step(); total+=loss.item()
    print(f"Epoch {epoch+1}/{epochs} loss={total/len(loader):.4f}")

In [ ]:
# Compare two test digits and display their embedding distance
test=datasets.MNIST("/content/data",train=False,download=True,transform=tfm)
def get_digit(label):
    idx=next(i for i,(_,y) in enumerate(test) if y==label)
    return test[idx][0], label
a,ya=get_digit(3); b,yb=get_digit(3); c,yc=get_digit(8)
model.eval()
with torch.no_grad():
    za,zb,zc=model.encoder(a[None].to(device)),model.encoder(b[None].to(device)),model.encoder(c[None].to(device))
    same=float(F.pairwise_distance(za,zb)); different=float(F.pairwise_distance(za,zc))
fig,ax=plt.subplots(1,3,figsize=(8,3))
for axis,img,title in zip(ax,[a,b,c],[f"Digit {ya}",f"Digit {yb}",f"Digit {yc}"]):
    axis.imshow(img.squeeze(),cmap="gray"); axis.set_title(title); axis.axis("off")
plt.show()
print(f"Embedding distance (same class): {same:.3f}")
print(f"Embedding distance (different class): {different:.3f}")
print("Lower distance means more similar in this learned embedding space; results vary with training.")

**Teaching point:** the same encoder processes both images, so its weights are shared. A distance threshold can make a match/no-match decision, but the threshold must be validated for the intended application.